# Proyecto Final – Análisis de Ventas Online + Indicadores Económicos

**Fuentes de datos:**
- Online Retail II (transacciones de una tienda online del Reino Unido, 2009–2011)
- World Bank Data (indicadores económicos por país y año)

**Notebooks del proyecto:**
- `01_carga_exploracion.ipynb` (este notebook) → secciones 1 a 4
- `02_analisis_descriptivo.ipynb` → secciones 5 a 9

## Índice de este notebook

1. **Carga y primer vistazo** – importación de ambos datasets, forma y tipos de datos
2. **Limpieza y transformación** – nulos, duplicados, tipos de datos, valores erróneos, nuevas columnas, corrección de nombres de país, unión (merge) de datasets
3. **Chequeo final de tipos de datos** – revisión y ajuste de tipos antes de exportar
4. **Exportación del dataset final** – guardado en `data/processed/`


## 1. Carga y primer vistazo

In [1]:
import pandas as pd

df_retail = pd.read_csv('../data/raw/online_retail_II.csv', encoding='ISO-8859-1')
df_econ = pd.read_csv('../data/raw/world_bank_data_2025.csv')

print("Retail:", df_retail.shape)
print("World Bank:", df_econ.shape)

Retail: (1067371, 8)
World Bank: (3472, 16)


### 1.1 Revisión de columnas y tipos de datos

In [2]:
# Retail
print("=== RETAIL ===")
print(df_retail.columns.tolist())
print(df_retail.dtypes)
df_retail.head()

=== RETAIL ===
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# World Bank
print("=== WORLD BANK ===")
print(df_econ.columns.tolist())
print(df_econ.dtypes)
df_econ.head()

=== WORLD BANK ===
['country_name', 'country_id', 'year', 'Inflation (CPI %)', 'GDP (Current USD)', 'GDP per Capita (Current USD)', 'Unemployment Rate (%)', 'Interest Rate (Real, %)', 'Inflation (GDP Deflator, %)', 'GDP Growth (% Annual)', 'Current Account Balance (% GDP)', 'Government Expense (% of GDP)', 'Government Revenue (% of GDP)', 'Tax Revenue (% of GDP)', 'Gross National Income (USD)', 'Public Debt (% of GDP)']
country_name                        object
country_id                          object
year                                 int64
Inflation (CPI %)                  float64
GDP (Current USD)                  float64
GDP per Capita (Current USD)       float64
Unemployment Rate (%)              float64
Interest Rate (Real, %)            float64
Inflation (GDP Deflator, %)        float64
GDP Growth (% Annual)              float64
Current Account Balance (% GDP)    float64
Government Expense (% of GDP)      float64
Government Revenue (% of GDP)      float64
Tax Revenue (% of

,country_name,country_id,year,Inflation (CPI %),GDP (Current USD),GDP per Capita (Current USD),Unemployment Rate (%),"Interest Rate (Real, %)","Inflation (GDP Deflator, %)",GDP Growth (% Annual),Current Account Balance (% GDP),Government Expense (% of GDP),Government Revenue (% of GDP),Tax Revenue (% of GDP),Gross National Income (USD),Public Debt (% of GDP)
0,Aruba,aw,2010,2.078141,2.453597e+09,24093.140151,NaN,11.666131,-1.223407,-2.733457,-18.752537,NaN,NaN,NaN,2.313385e+09,NaN
1,Aruba,aw,2011,4.316297,2.637859e+09,25712.384302,NaN,4.801974,4.005674,3.369237,-9.877656,NaN,NaN,NaN,2.391841e+09,NaN
2,Aruba,aw,2012,0.627472,2.615208e+09,25119.665545,NaN,8.200875,0.184033,-1.040800,3.473451,NaN,NaN,NaN,2.499118e+09,NaN
3,Aruba,aw,2013,-2.372065,2.727850e+09,25813.576727,NaN,10.709709,-1.995948,6.431483,-11.813206,NaN,NaN,NaN,2.563517e+09,NaN
4,Aruba,aw,2014,0.421441,2.790850e+09,26129.839062,NaN,3.213869,3.958897,-1.586575,-4.658577,NaN,NaN,NaN,2.688102e+09,NaN


### 1.2 Panorama general de valores nulos y duplicados

In [4]:
print("=== NULOS RETAIL ===")
print(df_retail.isnull().sum())
print("\nDuplicados retail:", df_retail.duplicated().sum())

print("\n=== NULOS WORLD BANK ===")
print(df_econ.isnull().sum())
print("\nDuplicados World Bank:", df_econ.duplicated().sum())

=== NULOS RETAIL ===
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Duplicados retail: 34335

=== NULOS WORLD BANK ===
country_name                          0
country_id                            0
year                                  0
Inflation (CPI %)                   778
GDP (Current USD)                   539
GDP per Capita (Current USD)        534
Unemployment Rate (%)               677
Interest Rate (Real, %)            1737
Inflation (GDP Deflator, %)         568
GDP Growth (% Annual)               560
Current Account Balance (% GDP)     909
Government Expense (% of GDP)      1652
Government Revenue (% of GDP)      1643
Tax Revenue (% of GDP)             1639
Gross National Income (USD)         676
Public Debt (% of GDP)             2620
dtype: int64

Duplicados World Bank: 0


### 1.3 Detección de valores erróneos y casos especiales

In [5]:
# Cantidades y precios negativos o cero
print("Quantity negativa o cero:", (df_retail['Quantity'] <= 0).sum())
print("Price negativo o cero:", (df_retail['Price'] <= 0).sum())

# Facturas canceladas (empiezan por 'C')
canceladas = df_retail['Invoice'].astype(str).str.startswith('C')
print("Facturas canceladas:", canceladas.sum())

Quantity negativa o cero: 22950
Price negativo o cero: 6207
Facturas canceladas: 19494


## 2. Limpieza y transformación

### 2.1 Eliminación de duplicados

In [6]:
print("Filas antes:", df_retail.shape[0])

df_retail = df_retail.drop_duplicates()

print("Filas después:", df_retail.shape[0])

Filas antes: 1067371
Filas después: 1033036


### 2.2 Separación de facturas canceladas

In [7]:
# Identificamos las cancelaciones
es_cancelacion = df_retail['Invoice'].astype(str).str.startswith('C')

# Guardamos las cancelaciones en un dataset aparte (para análisis futuro de devoluciones)
df_cancelaciones = df_retail[es_cancelacion].copy()

# Nos quedamos en df_retail solo con las ventas reales (no canceladas)
df_retail = df_retail[~es_cancelacion].copy()

print("Cancelaciones separadas:", df_cancelaciones.shape[0])
print("Retail (solo ventas):", df_retail.shape[0])

Cancelaciones separadas: 19104
Retail (solo ventas): 1013932


### 2.3 Eliminación de cantidades y precios inválidos

In [8]:
print("Filas antes:", df_retail.shape[0])

df_retail = df_retail[(df_retail['Quantity'] > 0) & (df_retail['Price'] > 0)]

print("Filas después:", df_retail.shape[0])

Filas antes: 1013932
Filas después: 1007913


### 2.4 Tratamiento de valores nulos

In [9]:
# Description: rellenamos con "Unknown"
df_retail['Description'] = df_retail['Description'].fillna('Unknown')

# Customer ID: creamos una columna que indique si el cliente está identificado
df_retail['Has_Customer_ID'] = df_retail['Customer ID'].notnull()

# Comprobación
print(df_retail['Description'].isnull().sum())  # debería dar 0
print(df_retail['Has_Customer_ID'].value_counts())

0
Has_Customer_ID
True     779425
False    228488
Name: count, dtype: int64


### 2.5 Conversión de fechas y creación de columnas temporales

In [10]:
df_retail['InvoiceDate'] = pd.to_datetime(df_retail['InvoiceDate'])

df_retail['Year'] = df_retail['InvoiceDate'].dt.year
df_retail['Month'] = df_retail['InvoiceDate'].dt.month
df_retail['Day'] = df_retail['InvoiceDate'].dt.day
df_retail['Weekday'] = df_retail['InvoiceDate'].dt.day_name()
df_retail['Quarter'] = df_retail['InvoiceDate'].dt.quarter

df_retail[['InvoiceDate', 'Year', 'Month', 'Day', 'Weekday', 'Quarter']].head()

,InvoiceDate,Year,Month,Day,Weekday,Quarter
0,2009-12-01 07:45:00,2009,12,1,Tuesday,4
1,2009-12-01 07:45:00,2009,12,1,Tuesday,4
2,2009-12-01 07:45:00,2009,12,1,Tuesday,4
3,2009-12-01 07:45:00,2009,12,1,Tuesday,4
4,2009-12-01 07:45:00,2009,12,1,Tuesday,4


### 2.6 Creación de la columna TotalPrice

In [11]:
df_retail['TotalPrice'] = df_retail['Quantity'] * df_retail['Price']

df_retail[['Quantity', 'Price', 'TotalPrice']].head()

,Quantity,Price,TotalPrice
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


### 2.7 Limpieza del dataset World Bank

In [12]:
# Eliminamos la columna con 75% de nulos
df_econ = df_econ.drop(columns=['Public Debt (% of GDP)'])

print(df_econ.shape)
df_econ.columns.tolist()

(3472, 15)


['country_name',
 'country_id',
 'year',
 'Inflation (CPI %)',
 'GDP (Current USD)',
 'GDP per Capita (Current USD)',
 'Unemployment Rate (%)',
 'Interest Rate (Real, %)',
 'Inflation (GDP Deflator, %)',
 'GDP Growth (% Annual)',
 'Current Account Balance (% GDP)',
 'Government Expense (% of GDP)',
 'Government Revenue (% of GDP)',
 'Tax Revenue (% of GDP)',
 'Gross National Income (USD)']

### 2.8 Comprobación de nombres de país antes del merge


In [13]:
paises_retail = set(df_retail['Country'].unique())
paises_econ = set(df_econ['country_name'].unique())

# Países que están en retail pero NO en econ (con esos nombres exactos)
print("En retail pero no en World Bank:")
print(sorted(paises_retail - paises_econ))

print("\nTotal países únicos en retail:", len(paises_retail))
print("Total países únicos en World Bank:", len(paises_econ))

En retail pero no en World Bank:
['Czech Republic', 'EIRE', 'European Community', 'Hong Kong', 'Korea', 'RSA', 'USA', 'Unspecified', 'West Indies']

Total países únicos en retail: 43
Total países únicos en World Bank: 217


### 2.9 Corrección de nombres de país


In [14]:
# Primero verificamos los nombres exactos que usa World Bank para los casos dudosos
print([p for p in paises_econ if 'Hong Kong' in p or 'Korea' in p or 'Czech' in p])

['Hong Kong SAR, China', 'Korea, Rep.', 'Czechia', "Korea, Dem. People's Rep."]


In [15]:
correccion_paises = {
    'Czech Republic': 'Czechia',
    'EIRE': 'Ireland',
    'Hong Kong': 'Hong Kong SAR, China',
    'Korea': 'Korea, Rep.',
    'RSA': 'South Africa',
    'USA': 'United States'
    # 'European Community', 'Unspecified' y 'West Indies' se quedan igual,
    # no tienen equivalente real de país y quedarán sin cruzar (NaN)
}

df_retail['Country'] = df_retail['Country'].replace(correccion_paises)

# Comprobación: cuántos países de retail siguen sin match ahora
paises_retail_actualizado = set(df_retail['Country'].unique())
print("Países en retail que aún no coinciden con World Bank:")
print(sorted(paises_retail_actualizado - paises_econ))

Países en retail que aún no coinciden con World Bank:
['European Community', 'Unspecified', 'West Indies']


### 2.10 Unión (merge) de los dos datasets

In [16]:
df_final = df_retail.merge(
    df_econ,
    left_on=['Country', 'Year'],
    right_on=['country_name', 'year'],
    how='left'
)

print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])
df_final.head()

Filas: 1007913
Columnas: 30


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Has_Customer_ID,Year,...,GDP per Capita (Current USD),Unemployment Rate (%),"Interest Rate (Real, %)","Inflation (GDP Deflator, %)",GDP Growth (% Annual),Current Account Balance (% GDP),Government Expense (% of GDP),Government Revenue (% of GDP),Tax Revenue (% of GDP),Gross National Income (USD)
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,True,2009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,True,2009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,True,2009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,True,2009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,True,2009,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.11 Revisión de nulos en el dataset final

In [17]:
print("Filas:", df_final.shape[0])
print("Columnas:", df_final.shape[1])
print()
print(df_final.isnull().sum())

Filas: 1007913
Columnas: 30

Invoice                                 0
StockCode                               0
Description                             0
Quantity                                0
InvoiceDate                             0
Price                                   0
Customer ID                        228488
Country                                 0
Has_Customer_ID                         0
Year                                    0
Month                                   0
Day                                     0
Weekday                                 0
Quarter                                 0
TotalPrice                              0
country_name                        44315
country_id                          44315
year                                44315
Inflation (CPI %)                   45861
GDP (Current USD)                   44315
GDP per Capita (Current USD)        44315
Unemployment Rate (%)               44349
Interest Rate (Real, %)            108945
Infla

### 2.12 Documentación de decisiones de limpieza

**Resumen de decisiones tomadas en esta fase:**

- Se eliminaron 34.335 filas duplicadas exactas.
- Se separaron 19.104 facturas canceladas en un dataset aparte (`df_cancelaciones`), al no representar ventas reales.
- Se eliminaron filas con `Quantity` o `Price` ≤ 0 (errores de registro).
- La columna `Description` (4.382 nulos) se rellenó con "Unknown".
- La columna `Customer ID` (228.488 nulos, ~23%) se mantuvo sin imputar, añadiendo la columna `Has_Customer_ID` para poder filtrar en análisis por cliente.
- Se eliminó la columna `Public Debt (% of GDP)` de World Bank por tener un 75% de valores nulos.
- Se corrigieron 6 nombres de país inconsistentes entre ambas fuentes (ej. "USA" → "United States") para maximizar coincidencias en el merge.
- Tras el merge, 44.315 filas (~4.4%) quedaron sin datos económicos, correspondientes a diciembre de 2009 (World Bank solo cubre desde 2010) y a 3 registros que no son países reales ("European Community", "Unspecified", "West Indies"). **Se decidió no imputar estos valores** para mantener la integridad de los datos, dejándolos como NaN.

## 3. Chequeo final de tipos de datos

In [18]:
df_final.dtypes

Invoice                                    object
StockCode                                  object
Description                                object
Quantity                                    int64
InvoiceDate                        datetime64[ns]
Price                                     float64
Customer ID                               float64
Country                                    object
Has_Customer_ID                              bool
Year                                        int32
Month                                       int32
Day                                         int32
Weekday                                    object
Quarter                                     int32
TotalPrice                                float64
country_name                               object
country_id                                 object
year                                      float64
Inflation (CPI %)                         float64
GDP (Current USD)                         float64


### 3.1 Ajustes de tipos de datos

In [19]:
# Customer ID: lo convertimos a texto (es un identificador, no una cantidad)
# Usamos Int64 (con mayúscula) como paso intermedio porque permite nulos, y luego a string
df_final['Customer ID'] = df_final['Customer ID'].astype('Int64').astype(str)
df_final['Customer ID'] = df_final['Customer ID'].replace('<NA>', None)

# Eliminamos 'year' y 'country_id' duplicados/redundantes (year = Year, country_name = Country)
df_final = df_final.drop(columns=['year', 'country_name', 'country_id'])

print(df_final.shape)
df_final.dtypes

(1007913, 27)


Invoice                                    object
StockCode                                  object
Description                                object
Quantity                                    int64
InvoiceDate                        datetime64[ns]
Price                                     float64
Customer ID                                object
Country                                    object
Has_Customer_ID                              bool
Year                                        int32
Month                                       int32
Day                                         int32
Weekday                                    object
Quarter                                     int32
TotalPrice                                float64
Inflation (CPI %)                         float64
GDP (Current USD)                         float64
GDP per Capita (Current USD)              float64
Unemployment Rate (%)                     float64
Interest Rate (Real, %)                   float64


## 4. Exportación del dataset final (checkpoint intermedio)

> Guardamos aquí el dataset limpio para no tener que repetir el proceso de limpieza en los siguientes notebooks de análisis.

In [20]:
df_final.to_csv('../data/processed/retail_final.csv', index=False)

print("Dataset guardado correctamente.")
print("Ubicación: data/processed/retail_final.csv")
print("Filas:", df_final.shape[0], "| Columnas:", df_final.shape[1])

Dataset guardado correctamente.
Ubicación: data/processed/retail_final.csv
Filas: 1007913 | Columnas: 27


---
## Fin del Notebook 1

Este notebook cubre las secciones 1 a 4: carga, limpieza, transformación y unión de las dos fuentes de datos.

El dataset limpio resultante se ha exportado a `data/processed/retail_final.csv` y será el punto de partida del siguiente notebook (secciones 5 a 9), dedicado al análisis descriptivo, estadístico y a la visualización de los datos.

**Próximo notebook:** `02_analisis_descriptivo.ipynb`